In [ ]:
import trimesh as tm
import pyvista as pv
import math as m
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy.spatial import ConvexHull, distance_matrix
import pandas as pd


In [ ]:
# Must match STATE in 2_reconstruct.ipynb. Was def3, hardcoded, against inputs
# named def31; both now follow the state that notebook actually reconstructed.
STATE = "def1"
meshes_orig=pv.read(f"ATAA_sint_{STATE}.stl_def.vtp")
meshes_def0=pv.read(f"ATAA_sint_{STATE}.stl_def0_cut.vtp")
Cline=pv.read(f"centerline_{STATE}.vtk")
meshes_def=meshes_orig.copy()
meshes_def=meshes_def.warp_by_vector("Displacement",factor=1)

meshes_orig.fill_holes(50,inplace=True)
meshes_def0.fill_holes(50,inplace=True)
meshes_def.fill_holes(50,inplace=True)

In [ ]:
max_plane= 85
min_plan= 30
plotter = pv.Plotter(notebook=True )
plotter.add_axes()

plotter.add_mesh(meshes_orig, color="g",opacity=0.2)
plotter.add_mesh(meshes_def0, color="r",opacity=0.5)
plotter.add_mesh(meshes_def,color="b",opacity=0.5)
plotter.add_mesh(Cline, color="b",opacity=1)
plotter.add_mesh(pv.Plane(center=Cline.points[min_plan],direction=Cline.points[min_plan+1]-Cline.points[min_plan],i_size=50,j_size=50), color="g",opacity=0.3)
plotter.add_mesh(pv.Plane(center=Cline.points[max_plane],direction=Cline.points[max_plane+1]-Cline.points[max_plane],i_size=50,j_size=50), color="g",opacity=0.3)
plotter.show(jupyter_backend='static')

DICE


In [ ]:
def repair_mesh(mesh):
    mesh.fix_normals()
    mesh.remove_unreferenced_vertices()
    mesh.update_faces(mesh.unique_faces())
    mesh.update_faces(mesh.nondegenerate_faces())
    if hasattr(mesh, "merge_vertices"):
        mesh.merge_vertices()
    # Try filling holes
    if not mesh.is_watertight:
        try:
            mesh = mesh.fill_holes(max_hole_size=np.inf) or mesh
        except:
            pass
    return mesh

def calculate_dice_coefficient(name_mesh1, name_mesh2,name):
    # Load the mesh files
    name_mesh1.save("temp1.stl")
    name_mesh2.save("temp2.stl")
    
    mesh1 = tm.load_mesh("temp1.stl")
    mesh2 = tm.load_mesh("temp2.stl")
    
    os.remove("temp1.stl")
    os.remove("temp2.stl")
    # Calculate volumes of each mesh
    volume1 = mesh1.volume
    volume2 = mesh2.volume
    print(f"Volume of {name[0]}: {int(volume1)}")
    print(f"Volume of {name[1]}: {int(volume2)}")
    
    # Ensure watertight meshes
    if not mesh1.is_volume:
        print("Mesh1 not watertight. Repairing...")
        mesh1 = repair_mesh(mesh1)
    
    if not mesh2.is_volume:
        print("Mesh2 not watertight. Repairing...")
        mesh2 = repair_mesh(mesh2)
    if not mesh1.is_volume or not mesh2.is_volume:
        print("Dice skipped: the generated surface mesh is not a closed volume at this density.")
        return None
    intersect = tm.boolean.intersection([mesh1, mesh2])
    print("intersec calculated")
    volume_int = intersect.volume

    if volume_int > (volume1 + volume2):
        print("Error: Intersection volume exceeds combined volume.")
        return None
    
    # Calculate Dice coefficient
    dice_coefficient = 2 * volume_int / (volume1 + volume2)
    print(f"Dice Coefficient: {dice_coefficient}")
    
    return dice_coefficient

calculate_dice_coefficient(meshes_def0,meshes_def ,["oring","mesured"])


In [ ]:
a1=np.float64(0.9969477761222635)
a2= np.float64(0.9964552618680977)
a3= np.float64(0.9954472300758308)
print(np.mean([a1,a2,a3]),np.std([a1,a2,a3]))

Diameter

In [ ]:


def cut_ring(mesh, cpoint, normal1,size=0.5):
    C_dist_1 = mesh.clip(normal=normal1, origin=cpoint + normal1 * size, invert=True)
    C_dist = C_dist_1.clip(normal=normal1, origin=cpoint - normal1 * size, invert=False)
    if len(C_dist.points) == 0:
        print("error")
        return None
    return C_dist

def cylinder_radius(points, center, normal):
    # Normalize the normal vector
    normal = normal / np.linalg.norm(normal)

    # Shift points relative to center
    shifted = points - center

    # Project onto plane: remove component along normal
    projected = shifted - np.outer(shifted @ normal, normal)

    # Compute distances in plane
    radii = np.linalg.norm(projected, axis=1)

    # Mean radius and diameter
    radius = np.mean(radii)
    diameter = 2 * radius
    return radius


plotter = pv.Plotter(notebook=True )
plotter.add_axes()
MaxD=[]
for i,point in enumerate(Cline.points[:-1]):
    if i>min_plan and i<max_plane and i % 2 == 0 :

        tring=cut_ring(meshes_def, point, Cline.points[i+1]-point,size=0.5)
        plotter.add_mesh(tring, color="r",opacity=1)
        if tring is None:
            continue
        MaxD.append(cylinder_radius(tring.points,point, Cline.points[i+1]-point))

print(MaxD)
print(np.mean(MaxD),np.std(MaxD))


plotter.add_mesh(meshes_def0, color="r",opacity=0.5)


plotter.show(jupyter_backend='static')

In [ ]:
#  Expected radi for each test
expected = np.array([16.0, 17.0, 18.0])

# Measured mean radi
measured = np.array([15.975955206677765,
                     16.96328095436241,
                     17.992399556340118])

# Reported standard deviations (errors)
errors = np.array([0.014321188753993408,
                   0.03225492101109946,
                   0.044229289496844265])

# Accuracy: difference between measured and expected
differences = measured - expected

# Precision: coefficient of variation (CV = std / mean)
cv = errors / measured

# Build results table
df = pd.DataFrame({
    "Test": [1, 2, 3],
    "Expected Diameter": expected,
    "Measured Diameter": measured,
    "Error (Std)": errors,
    "Difference (Accuracy)": differences,
    "CV (Error/Mean)": cv
})

# Summary statistics
mean_error = np.mean(errors)
std_error = np.std(errors, ddof=1)
min_error, max_error = np.min(errors), np.max(errors)

print("Precision and Accuracy Evaluation:\n")
print(df.round(4))
print("\nSummary:")
print(f"Average error: {mean_error:.4f}")
print(f"Error variability (std of errors): {std_error:.4f}")
print(f"Error range: {min_error:.4f} – {max_error:.4f}")

